# 全種 보전 자료 재검토 — 재현용 기록

## tl;dr
11,131종을 대조했다. NE 808종 중 360종에 참고 평가가 있으며 LC 332·NT 15·VU 9·EN 3·CR 1이다. 이는 현재 종 개념에 등급을 확정한 수가 아니다.

## Context & Methods
2026-10-09 고정 AviList v2025b와 IUCN GBIF 2026-1 원본 감사, 실제 TEST DB 조회 행의 최종 런타임 재현 결과를 읽는다. 실 HTTP 11,131회가 아니며 평가 원문 전수 독립 검증도 아니다.

### Key Assumptions
학명·명명자·평가 식별자 일치와 생물학적 종 범위 동등성은 구분한다. 과거 current_status는 HEAD 0689001의 baseline 정책이다. 최종 결과는 별도 replay 파일에 있다. 이 노트북은 저장된 원장을 재집계하며 서버를 조회하거나 수정하지 않는다.

## Data
저장소 루트 또는 하위 경로에서 실행한다. Python 표준 라이브러리만 필요하다. 원장은 gzip JSON이며 요약에 기록된 전체 SHA-256을 검사한다.

In [1]:
import gzip, hashlib, json
from pathlib import Path
from collections import Counter
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/robingraph').is_dir())
assets = root / 'docs/verification/assets'
def load_ledger(stem):
    summary = json.loads((assets / f'2026-10-09-{stem}-summary.json').read_text())
    raw = gzip.decompress((assets / summary['full_ledger']['path']).read_bytes())
    assert hashlib.sha256(raw).hexdigest() == summary['full_ledger']['uncompressed_sha256']
    value = json.loads(raw)
    assert len(value['species']) == summary['full_ledger']['rows'] == 11131
    return value
audit = load_ledger('conservation-reconciliation')
replay = load_ledger('conservation-runtime-replay')
print('Verified ledgers:', len(audit['species']), len(replay['species']))


Verified ledgers: 11131 11131


## Results
### NE 하위 집계 및 실제 DB 행 재현의 최종 상태

In [2]:
ne = [r for r in audit['species'] if r['taxonomy_category'] == 'NE']
assert len(ne) == 808
print(json.dumps(dict(Counter(r['ne_evidence_category'] for r in ne)), ensure_ascii=False, indent=2))
print(json.dumps(replay['statuses'], ensure_ascii=False, indent=2))
refs = [r for r in replay['species'] if r['reference_category']]
assert len(refs) == 360
assert all(r['category'] is None for r in refs)
assert not replay['issues']
print('Reference grades:', dict(Counter(r['reference_category'] for r in refs)))


{
  "unique_integrity_valid_name_authorship_reference_concept_unverified": 360,
  "no_exact_name_or_sis_reference": 330,
  "same_name_or_sis_reference_with_identity_or_integrity_gap": 118
}
{
  "linked_checklist": 7507,
  "needs_review": 807,
  "snapshot_only": 2816,
  "manual_override": 1
}
Reference grades: {'LC': 332, 'NT': 15, 'VU': 9, 'CR': 1, 'EN': 3}


### 큰부리까마귀와 위험 등급 참고 사례
현재 등급이 비어 있다는 사실과 참고 평가 자체가 있다는 사실을 동시에 확인한다.

In [3]:
for name in ['Corvus macrorhynchos', 'Pyrrhura subandina', 'Pica serica', 'Anser serrirostris']:
    row = next(r for r in replay['species'] if r['scientific_name'] == name)
    print(json.dumps(row, ensure_ascii=False))


{"taxon_id": "avilist-taxon:v2025b:20296", "scientific_name": "Corvus macrorhynchos", "taxonomy_category_raw": "NE", "category": null, "label": "평가 연결 확인 필요", "status": "needs_review", "reference_category": "LC", "reference_url": "https://www.iucnredlist.org/species/103727590/264280673"}
{"taxon_id": "avilist-taxon:v2025b:11626", "scientific_name": "Pyrrhura subandina", "taxonomy_category_raw": "NE", "category": null, "label": "평가 연결 확인 필요", "status": "needs_review", "reference_category": "CR", "reference_url": "https://www.iucnredlist.org/species/45422401/179484790"}
{"taxon_id": "avilist-taxon:v2025b:20193", "scientific_name": "Pica serica", "taxonomy_category_raw": "NE", "category": "LC", "label": "관심대상", "status": "manual_override", "reference_category": null, "reference_url": null}
{"taxon_id": "avilist-taxon:v2025b:344", "scientific_name": "Anser serrirostris", "taxonomy_category_raw": "NE", "category": null, "label": "평가 연결 확인 필요", "status": "needs_review", "reference_category":

## Takeaways
- 원자료 NE를 실제 미평가로 표시하지 않는다. 원값은 보존하고 확인되지 않은 유효 등급은 null이다.
- 참고 등급이 CR·EN·VU인 사례도 있어 일괄 LC 치환은 잘못이다.
- 기존 연결 7,507종, snapshot_only 2,816종, needs_review 807종, 기존 까치 manual_override 1종을 구분한다.
- 원본 감사 재생성은 scripts/audit_conservation_reconciliation.py, 실제 DB 행의 조회 재현은 assets/2026-10-09-verify-conservation-runtime.py를 사용한다. 고정 원본 파일은 별도 확보하고 해시를 확인한다.

검증 방식: 코드 셀을 작성 순서대로 별도 Python 프로세스에서 실행하여 출력과 assertion 결과를 저장했다. nbclient/Jupyter 커널은 이 환경에 없어 사용하지 않았으며 셀은 IPython 기능에 의존하지 않는다.